In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

zema = ZemaManager()


In [0]:
url = "https://ldcom365.sharepoint.com"

# WHEAT

In [0]:
start_date = datetime(2020, 1, 1)

end_date = datetime(2030, 1, 1)

In [0]:
arg = zema.get_curve(curve="P-CASH-LDC-INPUT-FLAT-WHEAT-AR-FOB Up River-11.5-USD-MT", period=f"{start_date}::{end_date}")
arg=arg[arg['observation']=='Last']
arg=arg[['date','value','contract_year','contract_month']]
arg['day']=arg['date'].dt.day
arg['month']=arg['date'].dt.month
arg['year']=arg['date'].dt.year


In [0]:
df=arg.copy()


# Convert to datetime if not already
df['date'] = pd.to_datetime(df['date'])

# --- STEP 1: Create a working day mapping from unique dates ---
unique_dates = df[['date']].drop_duplicates().copy()
unique_dates['Month'] = unique_dates['date'].dt.month
unique_dates['Year'] = unique_dates['date'].dt.year

# Working day = position of the date within the month/year
unique_dates['Working day'] = unique_dates.groupby(['Year', 'Month'])['date'].rank(method='dense').astype(int)

unique_dates['Season'] = unique_dates['date'].apply(lambda d: f"{d.year if d.month == 12 else d.year - 1}/{(d.year + 1) if d.month == 12 else d.year}")


# --- STEP 2: Merge working day and season into original df ---
df = df.merge(unique_dates[['date', 'Working day', 'Season', 'Month']], on='date', how='left')

# --- STEP 3: Pivot by date ---
pivot = df.pivot_table(
    index='date',
    columns='contract_month',
    values='value',
    aggfunc='first'
).reset_index()

# --- STEP 4: Merge metadata ---
meta = unique_dates[['date', 'Working day', 'Month', 'Season']]
df_final = pd.merge(meta, pivot, on='date')

# --- STEP 5: Rename contract month columns ---
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}
df_final.rename(columns=month_map, inplace=True)



# Add any missing months
for m in month_map.values():
    if m not in df_final.columns:
        df_final[m] = 0

# Reorder and format
cols = ['date', 'Working day', 'Month', 'Season'] + list(month_map.values())
df_final = df_final[cols].fillna(0)
df_final['date'] = df_final['date'].dt.strftime('%-d/%-m/%Y')  # e.g., 24/1/2025

month_order = ['Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov']
df_final = df_final[['date', 'Working day', 'Month', 'Season'] + month_order]



In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/prices/wheat_UPR.xlsx',df_final,index=False)

In [0]:
from pyspark.sql.functions import col, date_format, month, year, rank
from pyspark.sql.window import Window
import pandas as pd

# Convert the input DataFrame to a Spark DataFrame
df = spark.createDataFrame(arg)

# Convert to datetime if not already
df = df.withColumn('date', col('date').cast('timestamp'))

# --- STEP 1: Create a working day mapping from unique dates ---
unique_dates = df.select('date').distinct()
unique_dates = unique_dates.withColumn('Month', month(col('date')))
unique_dates = unique_dates.withColumn('Year', year(col('date')))

# Working day = position of the date within the month/year
window_spec = Window.partitionBy('Year', 'Month').orderBy('date')
unique_dates = unique_dates.withColumn('Working day', rank().over(window_spec))

unique_dates = unique_dates.withColumn('Season', 
    date_format(col('date'), "yyyy/MM").alias('Season'))

# --- STEP 2: Merge working day and season into original df ---
df = df.join(unique_dates, on='date', how='left')

# --- STEP 3: Pivot by date ---
pivot = df.groupBy('date').pivot('contract_month').agg({'value': 'first'})

# --- STEP 4: Merge metadata ---
meta = unique_dates.select('date', 'Working day', 'Month', 'Season')
df_final = meta.join(pivot, on='date')

# --- STEP 5: Rename contract month columns ---
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}
for old_name, new_name in month_map.items():
    df_final = df_final.withColumnRenamed(str(old_name), new_name)

# Add any missing months
for m in month_map.values():
    if m not in df_final.columns:
        df_final = df_final.withColumn(m, lit(0))

# Reorder and format
cols = ['date', 'Working day', 'Month', 'Season'] + list(month_map.values())
df_final = df_final.select(*cols).fillna(0)
df_final = df_final.withColumn('date', date_format(col('date'), 'd/M/yyyy'))

month_order = ['Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov']
df_final = df_final.select(['date', 'Working day', 'Month', 'Season'] + month_order)

display(df_final)

# CORN

In [0]:
start_date = datetime(2020, 1, 1)

end_date = datetime(2030, 1, 1)

In [0]:
arg_corn = zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
arg_corn=arg_corn[arg_corn['observation']=='Last']
arg_corn=arg_corn[['date','value','contract_year','contract_month']]
arg_corn['day']=arg_corn['date'].dt.day
arg_corn['month']=arg_corn['date'].dt.month
arg_corn['year']=arg_corn['date'].dt.year
arg_corn['date'] = pd.to_datetime(arg_corn['date'], dayfirst=True) 
arg_corn = arg_corn[arg_corn['date'] > '2020-01-31']

In [0]:
df_corn=arg_corn.copy()

# Convert to datetime if not already
df_corn['date'] = pd.to_datetime(df_corn['date'])

# --- STEP 1: Create a working day mapping from unique dates ---
unique_dates = df_corn[['date']].drop_duplicates().copy()
unique_dates['Month'] = unique_dates['date'].dt.month
unique_dates['Year'] = unique_dates['date'].dt.year

# Working day = position of the date within the month/year
unique_dates['Working day'] = unique_dates.groupby(['Year', 'Month'])['date'].rank(method='dense').astype(int)

unique_dates['Season'] = unique_dates['date'].apply(lambda d: f"{d.year}/{d.year + 1}" if d.month >= 3 else f"{d.year - 1}/{d.year}")


# --- STEP 2: Merge working day and season into original df_corn ---
df_corn = df_corn.merge(unique_dates[['date', 'Working day', 'Season', 'Month']], on='date', how='left')

# --- STEP 3: Pivot by date ---
pivot = df_corn.pivot_table(
    index='date',
    columns='contract_month',
    values='value',
    aggfunc='first'
).reset_index()

# --- STEP 4: Merge metadata ---
meta = unique_dates[['date', 'Working day', 'Month', 'Season']]
df_corn_final = pd.merge(meta, pivot, on='date')

# --- STEP 5: Rename contract month columns ---
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}
df_corn_final.rename(columns=month_map, inplace=True)



# Add any missing months
for m in month_map.values():
    if m not in df_corn_final.columns:
        df_corn_final[m] = 0

# Reorder and format
cols = ['date', 'Working day', 'Month', 'Season'] + list(month_map.values())
df_corn_final = df_corn_final[cols].fillna(0)
df_corn_final['date'] = df_corn_final['date'].dt.strftime('%-d/%-m/%Y')  # e.g., 24/1/2025

month_order = ['Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov','Dec', 'Jan', 'Feb']
df_corn_final = df_corn_final[['date', 'Working day', 'Month', 'Season'] + month_order]



In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/prices/corn_UPR_PREMIUM.xlsx',df_corn_final,index=False)